# Multi schema

## 3.4.1 状态类型
多种不同的状态源数据

不同的源数据被划分成四种，但是不要求agent包含4中状态
1. 全局状态：不进行区分，StateGraph传递给state_schema,一定要填写
2. 其他几个，为了使用langgraph更方便一点
3. 输入状态：（如果全局字段太多了，  给invoke的时候，不需要传入100个参数，只指定全局状态的一部分传入），给StateGraph的input_schema
2. 输出状态： result = graph.invoke... ， 不约束result会打印所有状态输出
3. 私有状态：不是直接声明传递给StateGraph的，  只是中间做临时补充逻辑， 不想更新全局的StateGraph大状态，节点与节点之间，私有状态即可， 进行补充

 ## 3.4.2 状态之间的关系

使用四种状态的时候，怎么符合官方的建议
1. 输入状态和输出状态是全局状态的子集，一种约束裁剪（但是底层源代码没设计死，不推荐这样做）
2. 私有状态，不能和全局状态重名，容易让人理解混乱
3. 定义node的时候， 入参和出参要写清楚状态（因为之前是不强制写状态的，内部dict会自动转换，更应该写清楚，提高可读性）
   1. 举例子，   第一个节点def node_1(state: InputState) -> OverAllState, 最后一个节点def node_199(state: OverStateAll) -> OutputState
4. 节点函数如果定义了InputState， 限制了属性A， 就不应该访问全局的B属性，会报错
5. 返回结果要保持一致结果，（虽然执行可能影响）

 ### 3.4.2.2 源码层面的约束
 LangGraph如何记录、裁剪和更新状态

1. langgraph底层，不是一个普通的字典，  而是拆分成多个可读写自己维护的状态机制， 每个状态底层对应Channel管道
2. 不同阶段记录在state图中。
3. 前三个状态，全局，输入，输出，在创建的时候已经记录了，创建图的时候，记录状态，后面添加节点会继续记录，记录在图中

什么时候记录：
1. 构建创建StateGraph（如果不填输入和输出，input和output等效于全局的）
2. 调用add_node，可能涉及**私有状态**，没有在全局状态里面的，也会添加到state图里面，扩展了，想要加一个节点，允许额外私有变量，不需要一开始全局声明



**如何访问**
1. 有input_schema就用它约束， 没有就用全局的约束， 为了约束调用的时候，外部船用的graph.invoke
2. def node_1(state: InputState) ， 节点 取决于state参数类型
3. def node... -> OverAllState， 节点返回的不需要完整OverAllState，只需要部分， 在全局状态有就行（会利用归约函数进行更新和合并） ，
4. 但是为了规范，推荐返回值和定义的返回值一致
4. 输出的状态什么时候用，得到result， 得到结果会根据out_put_schema会裁剪，  约束最终暴露的东西


In [1]:
# 上面讲完了四种状态的底层源码，现在写一写例子
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. 输入状态
class InputState(TypedDict):
    username: str

# 2. 输出状态
class OutputState(TypedDict):
    graph_output: str

# 3. 全局状态
class OverAllState(TypedDict):
    username: str
    graph_output: str
    nickname: str

# 4. 私有状态
class PrivateState(TypedDict):
    greeting: str

# 5. 第一个节点，对接start =》 InputState 修改的状态内容在全局状态中 =》 OverAllState
def node_1(state: InputState) -> OverAllState:
    return {
        "nickname": "Dear " + state["username"]
    }



# 输入要用上面对接的全局状态
# 6. 第二个节点， 对接node1 =》 OverAllState，使用的参数在全局状态中，修改的参数在私有状态中 =》 PrivateState
def node_2 (state: OverAllState) -> PrivateState:
    # 向私有状态添加greeting
    return {
        "greeting": "Hello, " + state["nickname"]
    }


# 7. 第三个节点， 地接node2 =》 PrivateState 使用的参数在私有状态中， 修改的状态在输出状态汇总
def node_3(state: PrivateState) -> OutputState: # 为了代码清晰，推荐最后写OutputState， 但不推荐瞎写OverAllState
    # 向输出状态添加最终的结果
    return {
        "graph_output": state["greeting"] + "很高兴认识你！"
    }


# 8. 构建状态图
builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState) # 在定义图的时候，加载全局状态，输入状态，输出状态

# 9. 添加节点
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2) #  添加节点的时候，添加私有状态
builder.add_node("node_3", node_3)

# 10. 添加边
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
# 填写的的输入状态
result = graph.invoke({"username": "atguigu"})

# 结果是输出状态
print(result) # 被定义OutputState约束， 只暴露graph_output属性， 被裁剪掉了


{'graph_output': 'Hello, Dear atguigu很高兴认识你！'}
